# The shapes a job comes in

Almost every job is one of a small number of shapes. Knowing which one you have
tells you what can go wrong, what can run at once, and what the graph is going
to cost you.

Nine shapes. Each one drawn, each with the twelve lines that build it.

| | shape | you have this when |
|---|---|---|
| 1 | **chain** | each step needs the one before it |
| 2 | **fan-out** | two things need the same input and not each other |
| 3 | **diamond** | they fan out and then meet again |
| 4 | **map** | the same work, once per item |
| 5 | **branch** | the data decides which way to go |
| 6 | **gate** | it should be allowed to say no |
| 7 | **fallback** | there is more than one way to get the same thing |
| 8 | **reuse** | the same little shape appears three times |
| 9 | **tournament** | you want several answers and then to pick one |

Nothing here downloads anything. Every cell runs in under a second.

In [1]:
try:
    import browsergraph  # noqa: F401
except ImportError:
    %pip install -q "browsergraph @ git+https://github.com/aidonerightcorp/browsergraph.git"

import json, pathlib
from dataclasses import replace

from browsergraph import execute, viz
from browsergraph.compile import compile_route
from browsergraph.manifest import NodeManifest, PortSpec
from browsergraph.workbench import Edge, NodeCandidate, StageDefinition, WorkbenchDefinition

# A fresh folder each run. Left-over files from a previous run make the "what
# did this produce" list a lie, and that list is half the point here.
import shutil
WORK = pathlib.Path("work")
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir()

# These come from the library rather than being redefined in every notebook.
# They used to be thirty lines pasted into each one, which meant anyone copying
# a notebook to start a project got helpers that did not exist in browsergraph.
from browsergraph.quick import chain, fanin, fanout, link, node, problems, step
from browsergraph.quick import graph as _graph
from browsergraph.quick import subgraph  # noqa: F401  (used by later notebooks)

# The notebooks kept the older names, and `build` also prints what is wrong
# rather than raising — in a notebook the complaint is the lesson.
stage = step

def build(title, task, stages, nodes, edges=()):
    bench = _graph(title, task, stages, nodes, edges)
    print("problems:", problems(bench) or "none")
    return bench

print("ready")

ready


One helper, so each shape below is only the interesting part. It builds the
graph, says whether anything is wrong with it, and draws it.

In [2]:
from browsergraph.quick import (chain, fanin, fanout, graph, link, node,
                                passthrough, step, subgraph)

def show(title, task, steps, nodes, links, note=""):
    """Build it, check it, count it, draw it."""
    bench = graph(title, task, steps, nodes, links)
    problems = bench.validate()
    print(f"{title}")
    print(f"  problems: {problems or 'none'}")
    print(f"  layers:   {bench.layers()}")
    print(f"  routes:   {bench.route_count():,}"
          + (f"   computations: {bench.computation_count():,}"
             if bench.computation_count() != bench.route_count() else ""))
    if note:
        print(f"  {note}")
    return bench

print("helper ready — nine shapes follow")

helper ready — nine shapes follow


## 1 · Chain

Each step needs the one before it. Nothing runs at the same time as anything
else, and one failure stops everything after it.

This is the shape most people draw first, and quite often it is wrong — not
because chains are bad, but because two of the steps did not actually need each
other and drawing them in a line threw that away.

In [3]:
nodes = [node("fetch.http", "fetch", gives=[("out", "Page")]),
         node("fetch.file", "fetch", gives=[("out", "Page")]),
         node("parse.html", "parse", [("in", "Page")], [("out", "Rows")]),
         node("save.csv",   "save",  [("in", "Rows")], [("out", "File")])]
steps = [step("fetch", "Get the page", [], [("out", "Page")], "fetch",
              ["fetch.http", "fetch.file"]),
         step("parse", "Pull the rows out", [("in", "Page")], [("out", "Rows")],
              "parse", ["parse.html"]),
         step("save",  "Write them down", [("in", "Rows")], [("out", "File")],
              "save", ["save.csv"])]

bench = show("Chain", "Fetch a page, parse it, save it.",
             steps, nodes, chain("fetch", "parse", "save"))
viz.dag(bench)

Chain
  problems: none
  layers:   [['fetch'], ['parse'], ['save']]
  routes:   2


Figure(svg='<svg viewBox="0 0 1100 226" width="1100" height="226" style="max-width:none" role="img"><defs><marker id="bg61727175-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-end"><path d="M0,0 L10,5 L0,10 z" fill="#8a93a0"/></marker></defs><text x="153.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 0</text><g><rect x="60" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="69" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Get the page</text><text x="69" y="108.0" font-size="9.5" fill="#68737f">2 candidates</text></g><text x="479.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 1</text><g><rect x="386" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="395" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Pull the rows out</text><text x="395" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="805.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 2</text><g><rect x="712" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="721" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Write them down</text><text x="721" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><path d="M246,100.0 C316.0,100.0 316.0,100.0 386,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg61727175-arrow)"/><path d="M572,100.0 C642.0,100.0 642.0,100.0 712,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg61727175-arrow)"/></svg>', title='Chain — shape', note='a chain. Boxes in the same layer are independent and may run together; every arrow is a typed port-to-port connection.', width=1100, height=226)

Three layers, one step in each. The layering is not a drawing choice — it is
worked out from which port feeds which, so a chain draws as a chain because it
*is* one.

## 2 · Fan-out

Two things need the same input and do not need each other. They can run at the
same time, and one failing does not stop the other.

The only thing that makes this shape possible is that neither one consumes the
other's output. Say that in the ports and it falls out for free.

In [4]:
nodes = [node("load.csv", "load", gives=[("out", "Table")]),
         node("stats.describe", "stats", [("in", "Table")], [("out", "Report")]),
         node("chart.hist",     "chart", [("in", "Table")], [("out", "Image")]),
         node("checks.nulls",   "checks",[("in", "Table")], [("out", "Report")])]
steps = [step("load",   "Load the table", [], [("out", "Table")], "load",
              ["load.csv"]),
         step("stats",  "Summarise it", [("in", "Table")], [("out", "Report")],
              "stats", ["stats.describe"]),
         step("chart",  "Draw it", [("in", "Table")], [("out", "Image")],
              "chart", ["chart.hist"]),
         step("checks", "Look for holes", [("in", "Table")], [("out", "Report")],
              "checks", ["checks.nulls"])]

bench = show("Fan-out", "Load once, then three independent readings.",
             steps, nodes, fanout("load", ["stats", "chart", "checks"]))
viz.dag(bench)

Fan-out
  problems: none
  layers:   [['load'], ['stats', 'chart', 'checks']]
  routes:   1


Figure(svg='<svg viewBox="0 0 1100 390" width="1100" height="390" style="max-width:none" role="img"><defs><marker id="bg89160946-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-end"><path d="M0,0 L10,5 L0,10 z" fill="#8a93a0"/></marker></defs><text x="153.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 0</text><g><rect x="60" y="156.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="69" y="175.0" font-size="11.5" font-weight="700" fill="#22303f">Load the table</text><text x="69" y="190.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="643.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 1 · 3 parallel</text><g><rect x="550" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="559" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Summarise it</text><text x="559" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><g><rect x="550" y="156.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="559" y="175.0" font-size="11.5" font-weight="700" fill="#22303f">Draw it</text><text x="559" y="190.0" font-size="9.5" fill="#68737f">1 candidate</text></g><g><rect x="550" y="238.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="559" y="257.0" font-size="11.5" font-weight="700" fill="#22303f">Look for holes</text><text x="559" y="272.0" font-size="9.5" fill="#68737f">1 candidate</text></g><path d="M246,182.0 C398.0,182.0 398.0,100.0 550,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg89160946-arrow)"/><path d="M246,182.0 C398.0,182.0 398.0,182.0 550,182.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg89160946-arrow)"/><path d="M246,182.0 C398.0,182.0 398.0,264.0 550,264.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg89160946-arrow)"/></svg>', title='Fan-out — shape', note='2 layers, widest 3. Boxes in the same layer are independent and may run together; every arrow is a typed port-to-port connection.', width=1100, height=390)

Three boxes in one layer. That is the graph telling you those three can run at
once — and `workers=3` is all it takes to make them.

## 3 · Diamond

They fan out and then meet again. The join is the part worth caring about: two
different things arrive at one step, and it has to be clear which is which.

That is why a join names the port each arrival lands on. An unlabelled join is
the single most common way a graph like this goes quietly wrong.

In [5]:
nodes = [node("load.csv", "load", gives=[("out", "Table")]),
         node("num.scale",  "num",  [("in", "Table")], [("out", "Numbers")]),
         node("cat.onehot", "cat",  [("in", "Table")], [("out", "Codes")]),
         node("join.concat","join", [("numbers", "Numbers"), ("codes", "Codes")],
              [("out", "Matrix")])]
steps = [step("load", "Load the table", [], [("out", "Table")], "load",
              ["load.csv"]),
         step("num",  "Scale the numbers", [("in", "Table")],
              [("out", "Numbers")], "num", ["num.scale"]),
         step("cat",  "Encode the words", [("in", "Table")], [("out", "Codes")],
              "cat", ["cat.onehot"]),
         step("join", "Put them together",
              [("numbers", "Numbers"), ("codes", "Codes")], [("out", "Matrix")],
              "join", ["join.concat"])]

links = [*fanout("load", ["num", "cat"]),
         # `fanin` takes a mapping, so you cannot write this wiring without
         # saying which port each side lands on.
         *fanin({"num": "numbers", "cat": "codes"}, "join")]

bench = show("Diamond", "Two treatments of one table, joined back up.",
             steps, nodes, links)
viz.dag(bench)

Diamond
  problems: none
  layers:   [['load'], ['num', 'cat'], ['join']]
  routes:   1


Figure(svg='<svg viewBox="0 0 1100 308" width="1100" height="308" style="max-width:none" role="img"><defs><marker id="bg41683367-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-end"><path d="M0,0 L10,5 L0,10 z" fill="#8a93a0"/></marker></defs><text x="153.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 0</text><g><rect x="60" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="69" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Load the table</text><text x="69" y="149.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="479.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 1 · 2 parallel</text><g><rect x="386" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="395" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Scale the numbers</text><text x="395" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><g><rect x="386" y="156.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="395" y="175.0" font-size="11.5" font-weight="700" fill="#22303f">Encode the words</text><text x="395" y="190.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="805.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 2</text><g><rect x="712" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="721" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Put them together</text><text x="721" y="149.0" font-size="9.5" fill="#68737f">1 candidate</text></g><path d="M246,141.0 C316.0,141.0 316.0,100.0 386,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg41683367-arrow)"/><path d="M246,141.0 C316.0,141.0 316.0,182.0 386,182.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg41683367-arrow)"/><path d="M572,100.0 C642.0,100.0 642.0,141.0 712,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg41683367-arrow)"/><text x="642.0" y="115.5" text-anchor="middle" font-size="9" fill="#68737f">numbers</text><path d="M572,182.0 C642.0,182.0 642.0,141.0 712,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg41683367-arrow)"/><text x="642.0" y="156.5" text-anchor="middle" font-size="9" fill="#68737f">codes</text></svg>', title='Diamond — shape', note='3 layers, widest 2. Boxes in the same layer are independent and may run together; every arrow is a typed port-to-port connection.', width=1100, height=308)

Look at the two arrows into `join`. Each one carries the name of the port it
arrives at. Swap them by mistake and the types disagree, and you are told before
anything runs rather than after the model trains on nonsense.

## 4 · Map

The same work, once per item. One step, many items, one collection back.

The step declares `List[Row]` and the node declares `Row`. That difference is
what makes it a map: the node handles one, the library handles all of them, and
nobody writes the loop.

In [6]:
nodes = [node("find.folder", "find", gives=[("out", "List[Path]")]),
         # One item in, one item out. The node has no idea there are others.
         node("read.one",  "read",  [("in", "Path")], [("out", "Row")]),
         node("total.sum", "total", [("in", "List[Row]")], [("out", "Report")])]
steps = [step("find",  "List the files", [], [("out", "List[Path]")], "find",
              ["find.folder"]),
         step("read",  "Read each one", [("in", "List[Path]")],
              [("out", "List[Row]")], "read", ["read.one"], kind="map"),
         step("total", "Add them up", [("in", "List[Row]")], [("out", "Report")],
              "total", ["total.sum"])]

bench = show("Map", "Read every file in a folder, then total them.",
             steps, nodes, chain("find", "read", "total"),
             note="the read step says List[Path]; its node says Path")
viz.dag(bench)

Map
  problems: none
  layers:   [['find'], ['read'], ['total']]
  routes:   1
  the read step says List[Path]; its node says Path


Figure(svg='<svg viewBox="0 0 1100 226" width="1100" height="226" style="max-width:none" role="img"><defs><marker id="bg2679288-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-end"><path d="M0,0 L10,5 L0,10 z" fill="#8a93a0"/></marker></defs><text x="153.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 0</text><g><rect x="60" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="69" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">List the files</text><text x="69" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="479.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 1</text><g><rect x="391" y="79.0" width="186" height="52" rx="7" fill="none" stroke="#8a93a0" stroke-width="1" opacity=".45"/><rect x="386" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="395" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Read each one</text><text x="563" y="93.0" text-anchor="end" font-size="9" font-weight="700" fill="#2d6cb5">MAP</text><text x="395" y="108.0" font-size="9.5" fill="#68737f">1 candidate · per item</text></g><text x="805.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 2</text><g><rect x="712" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="721" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Add them up</text><text x="721" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><path d="M246,100.0 C316.0,100.0 316.0,100.0 386,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg2679288-arrow)"/><path d="M572,100.0 C642.0,100.0 642.0,100.0 712,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg2679288-arrow)"/></svg>', title='Map — shape', note='a chain. A doubled outline runs once per item. Boxes in the same layer are independent and may run together; every arrow is a typed port-to-port connection.', width=1100, height=226)

The map step is drawn with a doubled edge and says MAP. One item failing is
reported as that item failing, with its index — not as "the batch failed", which
tells you nothing about which of four thousand files was the problem.

## 5 · Branch

The data decides which way to go. Only one way is taken; the other is skipped.

Skipped is a third outcome, and it needs to stay one. A path that never ran did
not fail, and did not succeed either.

In [7]:
nodes = [node("size.check", "size", [("in", "Table")],
              [("small", "Table"), ("large", "Table")]),
         node("quick.pandas", "quick", [("in", "Table")], [("out", "Result")]),
         node("heavy.chunked","heavy", [("in", "Table")], [("out", "Result")]),
         node("write.parquet","write", [("in", "Result")], [("out", "File")])]
steps = [step("size",  "How big is it?", [("in", "Table")],
              [("small", "Table"), ("large", "Table")], "size", ["size.check"],
              kind="branch"),
         step("quick", "Small: do it in memory", [("in", "Table")],
              [("out", "Result")], "quick", ["quick.pandas"]),
         step("heavy", "Large: do it in chunks", [("in", "Table")],
              [("out", "Result")], "heavy", ["heavy.chunked"]),
         step("write", "Write the answer", [("in", "Result")], [("out", "File")],
              "write", ["write.parquet"])]

links = [link("size", "quick", from_port="small"),
         link("size", "heavy", from_port="large"),
         # Both paths rejoin here, so `write` belongs to neither side and runs
         # whichever way the branch went.
         link("quick", "write"), link("heavy", "write")]

bench = show("Branch", "Choose a strategy from the size of the data.",
             steps, nodes, links)
viz.dag(bench)

Branch
  problems: none
  layers:   [['size'], ['quick', 'heavy'], ['write']]
  routes:   1   computations: 2


Figure(svg='<svg viewBox="0 0 1100 308" width="1100" height="308" style="max-width:none" role="img"><defs><marker id="bg36030356-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-end"><path d="M0,0 L10,5 L0,10 z" fill="#8a93a0"/></marker></defs><text x="153.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 0</text><g><rect x="60" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1" stroke-dasharray="6 3"/><text x="69" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">How big is it?</text><text x="237" y="134.0" text-anchor="end" font-size="9" font-weight="700" fill="#2d6cb5">BRANCH</text><text x="69" y="149.0" font-size="9.5" fill="#68737f">1 candidate · one way out</text></g><text x="479.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 1 · 2 parallel</text><g><rect x="386" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="395" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Small: do it in memory</text><text x="395" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><g><rect x="386" y="156.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="395" y="175.0" font-size="11.5" font-weight="700" fill="#22303f">Large: do it in chunks</text><text x="395" y="190.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="805.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 2</text><g><rect x="712" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="721" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Write the answer</text><text x="721" y="149.0" font-size="9.5" fill="#68737f">1 candidate</text></g><path d="M246,141.0 C316.0,141.0 316.0,100.0 386,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg36030356-arrow)"/><text x="316.0" y="115.5" text-anchor="middle" font-size="9" fill="#68737f">small</text><path d="M246,141.0 C316.0,141.0 316.0,182.0 386,182.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg36030356-arrow)"/><text x="316.0" y="156.5" text-anchor="middle" font-size="9" fill="#68737f">large</text><path d="M572,100.0 C642.0,100.0 642.0,141.0 712,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg36030356-arrow)"/><path d="M572,182.0 C642.0,182.0 642.0,141.0 712,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg36030356-arrow)"/></svg>', title='Branch — shape', note='3 layers, widest 2. A dashed outline takes only one way out. Boxes in the same layer are independent and may run together; every arrow is a typed port-to-port connection.', width=1100, height=308)

Two counts appeared under this one, and they are different questions.

**Routes — 1.** How many plans exist. There is one candidate per step, so there
is exactly one plan. A plan names a candidate for `heavy` even on a run that
goes small, because the choice is made before the data is seen.

**Computations — 2.** How many different things that one plan can be seen doing.
It can go small or it can go large, and those are not the same thing happening.

One plan, two behaviours. Add more candidates behind each side and it flips the
other way — plans that differ only behind the side that was not taken do the
same thing, so behaviours grow slower than plans.

## 6 · Gate

A step that is allowed to say no.

A quality check that cannot stop the run is a log line. If bad data flows on
regardless, the check did not check anything — it described.

In [8]:
nodes = [node("load.csv", "load", gives=[("out", "Table")]),
         node("gate.rules", "gate", [("in", "Table")],
              [("pass", "Table"), ("fail", "Report")]),
         node("use.model",  "use",   [("in", "Table")],  [("out", "Result")]),
         node("stop.report","stop",  [("in", "Report")], [("out", "Report")])]
steps = [step("load", "Load it", [], [("out", "Table")], "load", ["load.csv"]),
         step("gate", "Is it good enough?", [("in", "Table")],
              [("pass", "Table"), ("fail", "Report")], "gate", ["gate.rules"],
              kind="branch"),
         step("use",  "Use it", [("in", "Table")], [("out", "Result")], "use",
              ["use.model"]),
         step("stop", "Say why not", [("in", "Report")], [("out", "Report")],
              "stop", ["stop.report"])]

links = [link("load", "gate"),
         link("gate", "use",  from_port="pass"),
         link("gate", "stop", from_port="fail")]

bench = show("Gate", "Refuse to model data that failed its checks.",
             steps, nodes, links)
viz.dag(bench)

Gate
  problems: none
  layers:   [['load'], ['gate'], ['use', 'stop']]
  routes:   1   computations: 2


Figure(svg='<svg viewBox="0 0 1100 308" width="1100" height="308" style="max-width:none" role="img"><defs><marker id="bg94430396-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-end"><path d="M0,0 L10,5 L0,10 z" fill="#8a93a0"/></marker></defs><text x="153.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 0</text><g><rect x="60" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="69" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Load it</text><text x="69" y="149.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="479.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 1</text><g><rect x="386" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1" stroke-dasharray="6 3"/><text x="395" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Is it good enough?</text><text x="563" y="134.0" text-anchor="end" font-size="9" font-weight="700" fill="#2d6cb5">BRANCH</text><text x="395" y="149.0" font-size="9.5" fill="#68737f">1 candidate · one way out</text></g><text x="805.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 2 · 2 parallel</text><g><rect x="712" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="721" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Use it</text><text x="721" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><g><rect x="712" y="156.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="721" y="175.0" font-size="11.5" font-weight="700" fill="#22303f">Say why not</text><text x="721" y="190.0" font-size="9.5" fill="#68737f">1 candidate</text></g><path d="M246,141.0 C316.0,141.0 316.0,141.0 386,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg94430396-arrow)"/><path d="M572,141.0 C642.0,141.0 642.0,100.0 712,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg94430396-arrow)"/><text x="642.0" y="115.5" text-anchor="middle" font-size="9" fill="#68737f">pass</text><path d="M572,141.0 C642.0,141.0 642.0,182.0 712,182.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg94430396-arrow)"/><text x="642.0" y="156.5" text-anchor="middle" font-size="9" fill="#68737f">fail</text></svg>', title='Gate — shape', note='3 layers, widest 2. A dashed outline takes only one way out. Boxes in the same layer are independent and may run together; every arrow is a typed port-to-port connection.', width=1100, height=308)

A gate is a branch whose two sides are "carry on" and "stop and explain". Same
machinery, and worth its own name because it is the one people forget to build.

## 7 · Fallback

There is more than one way to get the same thing.

This one is not a shape in the graph at all — it is a shape in the *route*.
Several candidates on one step, each satisfying the same contract, and the
runner tries the next when the first fails.

In [9]:
nodes = [node("get.api",   "get", gives=[("out", "Data")], deterministic=False),
         node("get.scrape","get", gives=[("out", "Data")], deterministic=False),
         node("get.cache",  "get", gives=[("out", "Data")]),
         node("use.it", "use", [("in", "Data")], [("out", "Report")])]
steps = [step("get", "Get the data", [], [("out", "Data")], "get",
              ["get.api", "get.scrape", "get.cache"]),
         step("use", "Use it", [("in", "Data")], [("out", "Report")], "use",
              ["use.it"])]

bench = show("Fallback", "Three ways to get the same data.",
             steps, nodes, chain("get", "use"),
             note="one step, three candidates — the choice is a route, not a shape")
viz.route_space(bench, route={"get": "get.api", "use": "use.it"},
                alternative={"get": "get.cache", "use": "use.it"})

Fallback
  problems: none
  layers:   [['get'], ['use']]
  routes:   3
  one step, three candidates — the choice is a route, not a shape


Figure(svg='<svg viewBox="0 0 1180 260" width="1180" height="260" style="max-width:none" role="img"><style>.bg1128869-v{cursor:pointer}.bg1128869-v:hover rect{stroke:#c0392b;stroke-width:2}</style><polyline points="46,96 170,96 1010,96 1134,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,126 170,126 1010,96 1134,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,156 170,156 1010,96 1134,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,96 170,96 1010,96 1134,96" fill="none" stroke="#c0392b" stroke-width="2.6" opacity=".95"/><polyline points="46,156 170,156 1010,96 1134,96" fill="none" stroke="#c98a2b" stroke-width="2.6" opacity=".95" stroke-dasharray="7 4"/><line x1="108" y1="62" x2="108" y2="204" stroke="#dfe5ec" stroke-width="1"/><text x="108" y="46" text-anchor="middle" font-size="12" font-weight="700" fill="#22303f">Get the data</text><text x="108" y="58" text-anchor="middle" font-size="9" fill="#68737f">3 options</text><g class="bg1128869-v" data-stage="get" data-cid="get.api"><rect x="46" y="86" width="124" height="21" rx="5" fill="#fdeceb" stroke="#c0392b" stroke-width="1"/><text x="108" y="101" text-anchor="middle" font-size="10" fill="#22303f">api</text></g><g class="bg1128869-v" data-stage="get" data-cid="get.scrape"><rect x="46" y="116" width="124" height="21" rx="5" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="108" y="131" text-anchor="middle" font-size="10" fill="#22303f">scrape</text></g><g class="bg1128869-v" data-stage="get" data-cid="get.cache"><rect x="46" y="146" width="124" height="21" rx="5" fill="#fdf5e7" stroke="#c98a2b" stroke-width="1"/><text x="108" y="161" text-anchor="middle" font-size="10" fill="#22303f">cache</text></g><line x1="1072" y1="62" x2="1072" y2="204" stroke="#dfe5ec" stroke-width="1"/><text x="1072" y="46" text-anchor="middle" font-size="12" font-weight="700" fill="#22303f">Use it</text><text x="1072" y="58" text-anchor="middle" font-size="9" fill="#68737f">1 options</text><g class="bg1128869-v" data-stage="use" data-cid="use.it"><rect x="1010" y="86" width="124" height="21" rx="5" fill="#fdeceb" stroke="#c0392b" stroke-width="1"/><text x="1072" y="101" text-anchor="middle" font-size="10" fill="#22303f">it</text></g></svg>', title='Fallback — the space of routes', note='3 complete routes over 2 sub-steps.  Dashed amber: chosen with no evidence. Solid red: chosen after.', width=1180, height=260)

Drawn as a route space rather than a shape, because that is where the difference
lives. The graph is two boxes either way.

At run time you name the order:

```python
execute.run(plan, runtime, fallbacks={"get": ["get.scrape", "get.cache"]})
```

The step that fell back says so on its receipt, so "it worked" and "it worked on
the third try" stay distinguishable.

## 8 · Reuse

The same little shape appears three times.

Copy it and you have three things to keep in agreement. `subgraph` gives the
fragment a prefix and splices it in, so the ids cannot collide and there is one
definition.

In [10]:
# Written once.
QUALITY = [step("check",  "Check it", [("in", "Table")], [("out", "Report")],
                "check", ["check.rules", "check.stats"]),
           step("decide", "Good enough?", [("in", "Report")], [("out", "Table")],
                "decide", ["decide.threshold"])]
QUALITY_LINKS = chain("check", "decide")

inbound,  wiring_in  = subgraph("inbound",  QUALITY, QUALITY_LINKS)
outbound, wiring_out = subgraph("outbound", QUALITY, QUALITY_LINKS)

nodes = [node("load.csv", "load", gives=[("out", "Table")]),
         node("check.rules", "check", [("in", "Table")], [("out", "Report")]),
         node("check.stats", "check", [("in", "Table")], [("out", "Report")]),
         node("decide.threshold", "decide", [("in", "Report")], [("out", "Table")]),
         node("ship.write", "ship", [("in", "Table")], [("out", "File")])]

steps = [step("load", "Load it", [], [("out", "Table")], "load", ["load.csv"]),
         *inbound, *outbound,
         step("ship", "Ship it", [("in", "Table")], [("out", "File")], "ship",
              ["ship.write"])]

links = [link("load", "inbound.check"), *wiring_in,
         link("inbound.decide", "outbound.check"), *wiring_out,
         link("outbound.decide", "ship")]

bench = show("Reuse", "The same quality check, on the way in and on the way out.",
             steps, nodes, links,
             note="one definition, two copies, no id collisions")
viz.dag(bench)

Reuse
  problems: none
  layers:   [['load'], ['inbound.check'], ['inbound.decide'], ['outbound.check'], ['outbound.decide'], ['ship']]
  routes:   4
  one definition, two copies, no id collisions


Figure(svg='<svg viewBox="0 0 1170 226" width="1170" height="226" style="max-width:none" role="img"><defs><marker id="bg99836037-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-end"><path d="M0,0 L10,5 L0,10 z" fill="#8a93a0"/></marker></defs><text x="153.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 0</text><g><rect x="60" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="69" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Load it</text><text x="69" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="363.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 1</text><g><rect x="270" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="279" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Check it</text><text x="279" y="108.0" font-size="9.5" fill="#68737f">2 candidates</text></g><text x="573.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 2</text><g><rect x="480" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="489" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Good enough?</text><text x="489" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="783.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 3</text><g><rect x="690" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="699" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Check it</text><text x="699" y="108.0" font-size="9.5" fill="#68737f">2 candidates</text></g><text x="993.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 4</text><g><rect x="900" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="909" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Good enough?</text><text x="909" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="1203.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 5</text><g><rect x="1110" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="1119" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Ship it</text><text x="1119" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><path d="M246,100.0 C258.0,100.0 258.0,100.0 270,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg99836037-arrow)"/><path d="M456,100.0 C468.0,100.0 468.0,100.0 480,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg99836037-arrow)"/><path d="M666,100.0 C678.0,100.0 678.0,100.0 690,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg99836037-arrow)"/><path d="M876,100.0 C888.0,100.0 888.0,100.0 900,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg99836037-arrow)"/><path d="M1086,100.0 C1098.0,100.0 1098.0,100.0 1110,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg99836037-arrow)"/></svg>', title='Reuse — shape', note='a chain. Boxes in the same layer are independent and may run together; every arrow is a typed port-to-port connection.', width=1170, height=226)

`inbound.check` and `outbound.check` are separate steps with separate evidence —
the same check can be reliable on the way in and flaky on the way out, and you
would want to know that. What they share is the definition, not the history.

## 9 · Tournament

Get several answers, then pick one.

Different from a fallback. A fallback stops at the first thing that works; a
tournament runs them all on purpose, because you want to compare.

In [11]:
nodes = [node("load.csv", "load", gives=[("out", "Table")]),
         node("model.linear", "linear", [("in", "Table")], [("out", "Score")]),
         node("model.tree",   "tree",   [("in", "Table")], [("out", "Score")]),
         node("model.knn",    "knn",    [("in", "Table")], [("out", "Score")]),
         node("pick.best", "pick",
              [("a", "Score"), ("b", "Score"), ("c", "Score")],
              [("out", "Score")])]
steps = [step("load", "Load it", [], [("out", "Table")], "load", ["load.csv"]),
         step("linear", "Try a line", [("in", "Table")], [("out", "Score")],
              "linear", ["model.linear"]),
         step("tree",   "Try a tree", [("in", "Table")], [("out", "Score")],
              "tree", ["model.tree"]),
         step("knn",    "Try neighbours", [("in", "Table")], [("out", "Score")],
              "knn", ["model.knn"]),
         step("pick", "Keep the best",
              [("a", "Score"), ("b", "Score"), ("c", "Score")],
              [("out", "Score")], "pick", ["pick.best"])]

links = [*fanout("load", ["linear", "tree", "knn"]),
         *fanin({"linear": "a", "tree": "b", "knn": "c"}, "pick")]

bench = show("Tournament", "Three models, then keep the best.",
             steps, nodes, links)
viz.dag(bench)

Tournament
  problems: none
  layers:   [['load'], ['linear', 'tree', 'knn'], ['pick']]
  routes:   1


Figure(svg='<svg viewBox="0 0 1100 390" width="1100" height="390" style="max-width:none" role="img"><defs><marker id="bg78496905-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-end"><path d="M0,0 L10,5 L0,10 z" fill="#8a93a0"/></marker></defs><text x="153.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 0</text><g><rect x="60" y="156.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="69" y="175.0" font-size="11.5" font-weight="700" fill="#22303f">Load it</text><text x="69" y="190.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="479.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 1 · 3 parallel</text><g><rect x="386" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="395" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Try a line</text><text x="395" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><g><rect x="386" y="156.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="395" y="175.0" font-size="11.5" font-weight="700" fill="#22303f">Try a tree</text><text x="395" y="190.0" font-size="9.5" fill="#68737f">1 candidate</text></g><g><rect x="386" y="238.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="395" y="257.0" font-size="11.5" font-weight="700" fill="#22303f">Try neighbours</text><text x="395" y="272.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="805.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 2</text><g><rect x="712" y="156.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="721" y="175.0" font-size="11.5" font-weight="700" fill="#22303f">Keep the best</text><text x="721" y="190.0" font-size="9.5" fill="#68737f">1 candidate</text></g><path d="M246,182.0 C316.0,182.0 316.0,100.0 386,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg78496905-arrow)"/><path d="M246,182.0 C316.0,182.0 316.0,182.0 386,182.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg78496905-arrow)"/><path d="M246,182.0 C316.0,182.0 316.0,264.0 386,264.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg78496905-arrow)"/><path d="M572,100.0 C642.0,100.0 642.0,182.0 712,182.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg78496905-arrow)"/><text x="642.0" y="136.0" text-anchor="middle" font-size="9" fill="#68737f">a</text><path d="M572,182.0 C642.0,182.0 642.0,182.0 712,182.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg78496905-arrow)"/><text x="642.0" y="177.0" text-anchor="middle" font-size="9" fill="#68737f">b</text><path d="M572,264.0 C642.0,264.0 642.0,182.0 712,182.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg78496905-arrow)"/><text x="642.0" y="218.0" text-anchor="middle" font-size="9" fill="#68737f">c</text></svg>', title='Tournament — shape', note='3 layers, widest 3. Boxes in the same layer are independent and may run together; every arrow is a typed port-to-port connection.', width=1100, height=390)

Three in one layer, so they run together, and one step that sees all three.

Worth being honest about the cost: this runs every model every time. A search
over three candidates on one step would run one and learn which. Use a
tournament when you genuinely want all the answers — a report comparing them, or
an ensemble — and a search when you only want the winner.

---

## Picking your shape

Answer these in order and you will land on one:

1. **Does every step need the one before it?** → chain.
2. **Do two steps need the same input and not each other?** → fan-out. If they
   come back together, → diamond, and name the ports at the join.
3. **Is it the same work once per item?** → map. Write the node for one item.
4. **Does the data decide?** → branch. If one of the ways out is "stop", → gate.
5. **Is there more than one way to do a step?** → several candidates. Fallbacks
   at run time, a search when you want to find out which is best.
6. **Does the same fragment appear twice?** → subgraph with a prefix.
7. **Do you want all the answers, not just one?** → tournament.

Two things are worth writing on the wall:

**A loop is not on this list.** A loop with no argument for why it stops is how
a job runs forever. If you have one, either it is a map — the same work, once
per item, known in advance — or it needs a step that decides to stop, and that
step is a branch.

**A shape you drew as a chain that is really a diamond is the expensive
mistake.** Not because it fails, but because it quietly gives up all the
parallelism and every alternative route you could have had. That is what these
pictures are for: on paper the two look the same, and drawn they do not.